## 01\_telechargements

# 01 · Acquisition des données sources

| Fichier | Source | Taille |
| --- | --- | --- |
| `filosofi_2021_carreaux_200m_csv.zip`, décompressé dans `filosofi/` | INSEE, Filosofi 2021, carreaux de 200 m | environ 90 Mo |
| `arrets_france.csv` | Arrêts de transport en France (data.gouv.fr) | environ 440 Mo |
| `gtfs_astuce.zip` | GTFS du réseau Astuce (Métropole Rouen Normandie) | environ 5 Mo |
| `omphale_scenario_central.xlsx` | INSEE, projections de population Omphale, scénario central | environ 80 Ko |
| `epci_*.geojson`, `communes_*.geojson` | IGN ADMIN EXPRESS COG CARTO (Géoplateforme), Rouen et Strasbourg | < 1 Mo |

Les téléchargements sont idempotents : un fichier déjà présent et de taille plausible n'est pas retéléchargé.

**Sorties :** `Files/brut/`.

In [ ]:
# 1. Dossiers de travail
import os, json, time, zipfile, urllib.request, urllib.parse

BASE = "/lakehouse/default/Files"
if not os.path.isdir(BASE):
    raise RuntimeError(
        "Lakehouse par défaut non attaché : attacher le Lakehouse mobilite au notebook."
    )
BRUT = os.path.join(BASE, "brut")
os.makedirs(BRUT, exist_ok=True)
ENTETES = {"User-Agent": "hackathon-mobilite (Microsoft Fabric)"}
print("Dossier de sortie :", BRUT)

Dossier de sortie : /lakehouse/default/Files/brut


In [ ]:
# 2. Fonction de téléchargement
def telecharger(urls, nom, taille_min_mo):
    """Essaie chaque lien dans l'ordre. Ne refait rien si le fichier est déjà là."""
    chemin = os.path.join(BRUT, nom)
    if os.path.exists(chemin) and os.path.getsize(chemin) >= taille_min_mo * 1e6:
        print(f"Déjà présent : {nom} ({os.path.getsize(chemin) / 1e6:.1f} Mo)")
        return chemin
    if isinstance(urls, str):
        urls = [urls]
    for url in urls:
        for tentative in range(3):
            try:
                debut = time.time()
                requete = urllib.request.Request(url, headers=ENTETES)
                with urllib.request.urlopen(requete, timeout=120) as r, open(chemin, "wb") as f:
                    while True:
                        bloc = r.read(1 << 20)
                        if not bloc:
                            break
                        f.write(bloc)
                taille = os.path.getsize(chemin)
                if taille < taille_min_mo * 1e6:
                    raise RuntimeError(f"fichier trop petit ({taille / 1e6:.2f} Mo)")
                print(f"OK : {nom} ({taille / 1e6:.1f} Mo en {time.time() - debut:.0f} s)")
                return chemin
            except Exception as e:
                print(f"  Essai {tentative + 1} raté pour {nom} : {e}")
                if os.path.exists(chemin):
                    os.remove(chemin)
                time.sleep(3 * (tentative + 1))
    print(f"ÉCHEC : {nom}. Source : {urls[0]} (dépôt manuel possible dans Files/brut/{nom}).")
    return None

In [ ]:
# 3. Les fichiers à télécharger
fichiers = {}
fichiers["filosofi"] = telecharger(
    "https://www.insee.fr/fr/statistiques/fichier/8735162/Filosofi2021_carreaux_200m_csv.zip",
    "filosofi_2021_carreaux_200m_csv.zip", 50)
fichiers["arrets"] = telecharger(
    "https://www.data.gouv.fr/api/1/datasets/r/69cf54c6-6591-4920-b1d6-2a5292964606",
    "arrets_france.csv", 200)
fichiers["astuce"] = telecharger(
    ["https://api.mrn.cityway.fr/dataflow/offre-tc/download?provider=ASTUCE&dataFormat=gtfs&dataProfil=ASTUCE",
     # copie archivée par transport.data.gouv.fr le 18/10/2026, en secours
     "https://transport-data-gouv-fr-resource-history-prod.cellar-c2.services.clever-cloud.com/64973/64973.20261018.181531.849363.zip"],
    "gtfs_astuce.zip", 1)
fichiers["omphale"] = telecharger(
    "https://www.insee.fr/fr/statistiques/fichier/6652134/projections_scenario_central.xlsx",
    "omphale_scenario_central.xlsx", 0.02)

Déjà présent : filosofi_2021_carreaux_200m_csv.zip (91.6 Mo)
Déjà présent : arrets_france.csv (437.0 Mo)
Déjà présent : gtfs_astuce.zip (5.5 Mo)
Déjà présent : omphale_scenario_central.xlsx (0.1 Mo)


In [ ]:
# 4. Décompresser Filosofi et vérifier le GTFS
if fichiers["filosofi"]:
    dossier = os.path.join(BRUT, "filosofi")
    os.makedirs(dossier, exist_ok=True)
    with zipfile.ZipFile(fichiers["filosofi"]) as z:
        for info in z.infolist():
            print(f"   {info.filename} ({info.file_size / 1e6:.1f} Mo)")
        z.extractall(dossier)
    print("Filosofi décompressé dans", dossier)

if fichiers["astuce"]:
    with zipfile.ZipFile(fichiers["astuce"]) as z:
        noms = z.namelist()
    attendus = ["stops.txt", "routes.txt", "trips.txt", "stop_times.txt"]
    manquants = [n for n in attendus if not any(x.endswith(n) for x in noms)]
    print("GTFS Astuce complet." if not manquants else f"GTFS Astuce incomplet, il manque : {manquants}")

   carreaux_200m_mart.csv (2.2 Mo)
   carreaux_200m_met.csv (467.7 Mo)
   carreaux_200m_reun.csv (2.9 Mo)
Filosofi décompressé dans /lakehouse/default/Files/brut/filosofi
GTFS Astuce complet.


In [ ]:
# 5. Contours IGN (ADMIN EXPRESS COG CARTO), lus sur le service en ligne de la Géoplateforme
WFS = "https://data.geopf.fr/wfs/ows"

def wfs(couche, **parametres):
    requete = {"SERVICE": "WFS", "VERSION": "2.0.0", "REQUEST": "GetFeature", "TYPENAMES": couche,
               "OUTPUTFORMAT": "application/json", "SRSNAME": "EPSG:4326", "COUNT": "5000"}
    requete.update(parametres)
    url = WFS + "?" + urllib.parse.urlencode(requete)
    with urllib.request.urlopen(urllib.request.Request(url, headers=ENTETES), timeout=120) as r:
        return json.loads(r.read().decode("utf-8"))

def bornes(geometrie):
    """Rectangle (lat_min, lon_min, lat_max, lon_max) qui contient une géométrie GeoJSON."""
    points = []
    def parcourir(c):
        if isinstance(c[0], (int, float)):
            points.append(c)
        else:
            for x in c:
                parcourir(x)
    parcourir(geometrie["coordinates"])
    lons = [p[0] for p in points]
    lats = [p[1] for p in points]
    return min(lats), min(lons), max(lats), max(lons)

def enregistrer_geojson(objets, nom):
    with open(os.path.join(BRUT, nom), "w", encoding="utf-8") as f:
        json.dump({"type": "FeatureCollection", "features": objets}, f)
    print(f"OK : {nom} ({len(objets)} objets)")

TERRITOIRES = {  # nom court : (latitude et longitude du centre-ville, mot à trouver dans le nom de l'EPCI)
    "rouen": (49.440, 1.093, "Rouen"),
    "strasbourg": (48.582, 7.745, "Strasbourg"),
}
fichiers["contours"] = True
for court, (lat, lon, mot) in TERRITOIRES.items():
    try:
        cadre = f"{lat - 0.005},{lon - 0.008},{lat + 0.005},{lon + 0.008},urn:ogc:def:crs:EPSG::4326"
        epcis = wfs("ADMINEXPRESS-COG-CARTO.LATEST:epci", BBOX=cadre)["features"]
        epci = next(f for f in epcis if mot.lower() in f["properties"].get("nom_officiel", "").lower())
        siren = epci["properties"]["code_siren"]
        membres = set(epci["properties"]["codes_insee_des_communes_membres"].split("/"))
        print(f"{epci['properties']['nom_officiel']} : SIREN {siren}, {len(membres)} communes")
        enregistrer_geojson([epci], f"epci_{court}.geojson")
        try:
            communes = wfs("ADMINEXPRESS-COG-CARTO.LATEST:commune",
                           CQL_FILTER=f"codes_siren_des_epci LIKE '%{siren}%'")["features"]
        except Exception:
            communes = []
        if len(communes) != len(membres):   # repli : communes du rectangle englobant, filtrées par code INSEE
            la1, lo1, la2, lo2 = bornes(epci["geometry"])
            zone = wfs("ADMINEXPRESS-COG-CARTO.LATEST:commune",
                       BBOX=f"{la1},{lo1},{la2},{lo2},urn:ogc:def:crs:EPSG::4326")["features"]
            communes = [c for c in zone if c["properties"].get("code_insee") in membres]
        enregistrer_geojson(communes, f"communes_{court}.geojson")
    except Exception as e:
        fichiers["contours"] = None
        print(f"ÉCHEC des contours {court} : {e}. Repli possible : archive ADMIN EXPRESS déposée dans Files/brut.")

Métropole Rouen Normandie : SIREN 200023414, 71 communes
OK : epci_rouen.geojson (1 objets)
OK : communes_rouen.geojson (71 objets)
Eurométropole de Strasbourg : SIREN 246700488, 33 communes
OK : epci_strasbourg.geojson (1 objets)
OK : communes_strasbourg.geojson (33 objets)


In [ ]:
# 6. Récapitulatif
for racine, _, noms in os.walk(BRUT):
    if "releves" in racine:
        continue
    for nom in sorted(noms):
        chemin = os.path.join(racine, nom)
        print(f"{os.path.relpath(chemin, BASE):60s} {os.path.getsize(chemin) / 1e6:8.2f} Mo")
manquants = [k for k, v in fichiers.items() if not v]
print("\nTout est là." if not manquants else f"\nÀ récupérer à la main : {manquants}")

brut/arrets_france.csv                                         436.97 Mo
brut/communes_rouen.geojson                                      0.30 Mo
brut/communes_strasbourg.geojson                                 0.17 Mo
brut/epci_rouen.geojson                                          0.05 Mo
brut/epci_strasbourg.geojson                                     0.03 Mo
brut/filosofi_2021_carreaux_200m_csv.zip                        91.61 Mo
brut/gtfs_astuce.zip                                             5.46 Mo
brut/omphale_scenario_central.xlsx                               0.08 Mo
brut/filosofi/carreaux_200m_mart.csv                             2.23 Mo
brut/filosofi/carreaux_200m_met.csv                            467.72 Mo
brut/filosofi/carreaux_200m_reun.csv                             2.89 Mo
brut/citiz_manuel/Pedestrian and road network (OpenStreetMap)/haute-normandie-261019-free.gpkg/README     0.00 Mo
brut/citiz_manuel/Pedestrian and road network (OpenStreetMap)/haute-normandie-26101